# معمل بناء أول وكيل ذكاء اصطناعي موحد عبر `create_agent`

أهلاً بك في المعمل العملي للمحاضرة 59!

في المعمل ده هنجرب عملياً إزاي نبني أول AI Agent في LangChain v1+ باستخدام دالة `create_agent` الموحدة:
1. نشوف الفرق بين وكيل من غير أدوات ووكيل متوصل بأداة الطقس.
2. نفهم إزاي الموديل بيقرا الـ `Docstring` بتاع الدالة عشان يعرف يختارها.
3. نستدعي الوكيل ونشوف رحلة الرسائل الأربعة لحد ما يرجع لنا الإجابة!


In [29]:
# loaded the liberary and the model successfully, now let's create a unified agent that can use multiple tools to get weather information from Open-Meteo API.
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_groq import ChatGroq

load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("Model loaded successfully!")


Model loaded successfully!


In [30]:
# tools to get weather information from Open-Meteo API
import requests
from langchain.tools import tool
WMO_CODES = {
    0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
    45: "fog", 48: "freezing fog",
    51: "light drizzle", 53: "drizzle", 55: "heavy drizzle",
    61: "light rain", 63: "rain", 65: "heavy rain",
    71: "light snow", 73: "snow", 75: "heavy snow",
    80: "rain showers", 81: "heavy rain showers", 82: "violent rain showers",
    95: "thunderstorm", 96: "thunderstorm with hail", 99: "severe thunderstorm with hail",
}

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    try:
        # 1) City name -> coordinates
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=10,
        )
        geo.raise_for_status()
        results = geo.json().get("results")
        if not results:
            return f"Couldn't find a city named '{city}'."

        place = results[0]
        name = f"{place['name']}, {place.get('country', '')}".strip(", ")

        # 2) Coordinates -> current weather
        wx = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
            },
            timeout=10,
        )
        wx.raise_for_status()
        cur = wx.json()["current"]

        condition = WMO_CODES.get(cur["weather_code"], "unknown conditions")
        return (
            f"Weather in {name}: {condition}, {cur['temperature_2m']}°C, "
            f"humidity {cur['relative_humidity_2m']}%, wind {cur['wind_speed_10m']} km/h."
        )
    except requests.RequestException as e:
        return f"Failed to fetch weather for '{city}': {e}"


if __name__ == "__main__":
    print(get_weather.invoke({"city": "Tanta"}))

Weather in Tanta, Egypt: mainly clear, 21.9°C, humidity 43%, wind 13.8 km/h.


In [31]:
real_agent = create_agent(
    model=llm,
    tools=[get_weather],
   system_prompt="You are a helpful assistant. For weather questions, call the get_weather tool with the city name, then summarize the result."
)
response = real_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather in Cairo?"}]  
})



for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

What is the weather in Cairo?
================================== Ai Message ==================================
Tool Calls:
  get_weather (fc_7826a13c-498d-4d3e-86b3-1ce36a6b4039)
 Call ID: fc_7826a13c-498d-4d3e-86b3-1ce36a6b4039
  Args:
    city: Cairo
================================= Tool Message =================================
Name: get_weather

Weather in Cairo, Egypt: mainly clear, 21.0°C, humidity 46%, wind 14.2 km/h.
================================== Ai Message ==================================

**Weather in Cairo (today)**  
- **Condition:** Mainly clear  
- **Temperature:** 21 °C  
- **Humidity:** 46 %  
- **Wind:** 14.2 km/h  

Enjoy your day!
